# 02. Small GPT Training Pipeline & Optimization
**Project:** From Small GPT to an Agentic AI System  
**Objective:** Implement a compact causal Transformer from scratch in PyTorch, train with Cosine LR scheduling and mixed precision on Colab GPU, and save checkpoints.

In [ ]:
# Setup & Dependencies
!pip install -q tokenizers transformers accelerate matplotlib
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Model Configuration & Architecture

In [ ]:
from src.model.config import GPTConfig
from src.model.transformer import SmallGPT

# Configure a 6.2M parameter GPT-Mini
config = GPTConfig(
    vocab_size=500,
    block_size=256,
    n_layer=6,
    n_head=6,
    n_embd=288,
    dropout=0.1,
    tie_weights=True
)
model = SmallGPT(config).to(device)
print(f"Initialized SmallGPT with {model.get_num_params():,} non-embedding parameters.")

## 2. Dataset Preparation & Dataloaders

In [ ]:
from src.data.dataset import generate_synthetic_code_corpus, build_and_split_corpus, CausalLMDataset
from src.data.tokenizer import BPETokenizerWrapper
from torch.utils.data import DataLoader

corpus = generate_synthetic_code_corpus(num_samples=1000, seed=42)
train_set, val_set, test_set = build_and_split_corpus(corpus)
tokenizer = BPETokenizerWrapper(vocab_size=500).train_from_iterator(corpus)

train_tokens = [tok for t in train_set for tok in tokenizer.encode(t, add_special_tokens=True)]
val_tokens = [tok for t in val_set for tok in tokenizer.encode(t, add_special_tokens=True)]

train_loader = DataLoader(CausalLMDataset(train_tokens, block_size=config.block_size), batch_size=16, shuffle=True)
val_loader = DataLoader(CausalLMDataset(val_tokens, block_size=config.block_size), batch_size=16, shuffle=False)
print(f"Train chunks: {len(train_loader)} | Val chunks: {len(val_loader)}")

## 3. Training Loop with Mixed Precision & Cosine Warmup

In [ ]:
from src.training.trainer import GPTTrainer, TrainingArgs

args = TrainingArgs(
    learning_rate=5e-4,
    min_learning_rate=5e-5,
    warmup_steps=30,
    max_steps=200,
    eval_interval=50,
    batch_size=16,
    device=device,
    checkpoint_dir="checkpoints/colab_run"
)
trainer = GPTTrainer(model, args, train_loader, val_loader)
summary = trainer.train()

## 4. Plotting Loss Convergence

In [ ]:
import matplotlib.pyplot as plt
history = summary["history"]
steps = [h["step"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]

plt.figure(figsize=(8, 4))
plt.plot(steps, train_losses, label="Train Loss", linestyle="--")
plt.plot(steps, val_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Step")
plt.ylabel("Cross-Entropy Loss")
plt.title("SmallGPT Training Loss Curve")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()